In [25]:
# youtube - https://www.youtube.com/watch?v=zCEJurLGFRk&t=14s
# % pip install google-api-python-client google-auth-httplib2 google-auth-oauthlib gspread
import gspread
from google.oauth2.service_account import Credentials
import pandas as pd
from gspread_dataframe import set_with_dataframe

from data import Data
from cache_pandas import timed_lru_cache
import logging

@timed_lru_cache(seconds=None, maxsize=None)
def dataF():
    logging.info("Fetching data in dataF()")
    df = Data()
    vix_data = df.vix_history()
    policy_rate1, policy_rate2, policy_rate3 = df.policy_rate()
    fx = df.forex_exchange()
    cds = df.cds()
    liquidity = df.liquidity()
    gdp_growth = df.gdp_growth()
    logging.info("Completed fetching data in dataF()")
    return (
        vix_data, #use this
        policy_rate1, #use this
        policy_rate2,
        policy_rate3,
        fx, #use this
        cds, #use this
        liquidity, #use this
        gdp_growth, #use this
    )


vix_data, policy_rate1, policy_rate2, policy_rate3, fx, cds, liquidity, gdp_growth, = dataF()

scopes = [
    'https://www.googleapis.com/auth/spreadsheets']

creds = Credentials.from_service_account_file(
    'credential.json', scopes=scopes)

client = gspread.authorize(creds)

# 1PP6gpBcoOHjjgCx7LuHLa3dv4ET6ufKvpSY4UDvBczQ
sheet_id = '11Ora6_5EoQJdgnUpjjZgFZyrILguo1c32mde_uQwupw'
sheet = client.open_by_key(sheet_id)

df = pd.DataFrame({
    "Name": ["Alice", "Bob", "Charlie"],
    "Age": [25, 30, 40]
})

set_with_dataframe(sheet.sheet1, df) 

values_list = sheet.sheet1.get_all_values()
print(values_list)

worksheet_list = map(lambda x: x.title, sheet.worksheets())
worksheet_name = {'vix_data': vix_data, 'policy_rate1': policy_rate1, 'fx': fx, 'cds': cds, 'liquidity': liquidity, 'gdp_growth': gdp_growth}
for new_worksheet_name, data in worksheet_name.items():
    if new_worksheet_name in worksheet_list:
        sheet1 = sheet.worksheet(new_worksheet_name)
    else:
        sheet1 = sheet.add_worksheet(title=new_worksheet_name, rows=10, cols=10)
    set_with_dataframe(sheet1, data)

# sheet.worksheet(new_worksheet_name) -- to select worksheet page
# sheet.add_worksheet(new_worksheet_name, rows=10, cols=10) -- add new worksheet page

c:\Users\AhmadAizudeen\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Desktop\NLP Project\cfm-nlp\data.py:161: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date"] = pd.to_datetime(df["Date"]) + pd.offsets.MonthEnd(0)
c:\Users\AhmadAizudeen\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Desktop\NLP Project\cfm-nlp\data.py:296: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df2["Date"] = pd.to_datetime(df2["Date"]) + pd.offsets.MonthEnd(0)
c:\USERS\AHMADAIZUDEEN\ONEDRIVE - THE SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\DESKTOP\NLP PROJECT\VENC\Lib\site-packages\pandas\core\indexes\base.py:7588: FutureWarning: Dtype inference on a p

[['Name', 'Age'], ['Alice', '25'], ['Bob', '30'], ['Charlie', '40']]


In [ ]:
# 
# Loop through worksheet names and data
for new_worksheet_name, data in worksheet_name.items():
    if new_worksheet_name in worksheet_list:
        sheet = spreadsheet.worksheet(new_worksheet_name)  # Select existing worksheet
    else:
        sheet = spreadsheet.add_worksheet(title=new_worksheet_name, rows=10, cols=10)  # ✅ Use 'spreadsheet' object
    
    # Upload DataFrame to Google Sheets
    set_with_dataframe(sheet, data)

print("All data uploaded successfully!")

AttributeError: 'Worksheet' object has no attribute 'add_worksheet'

In [ ]:
worksheet_list = map(lambda x: x.title, sheet.worksheets())
worksheet_name = {'vix_data': vix_data, 'policy_rate1': policy_rate1, 'fx': fx, 'cds': cds, 'liquidity': liquidity, 'gdp_growth': gdp_growth}
for new_worksheet_name, data in worksheet_name.items():
    if new_worksheet_name in worksheet_list:
        sheet = sheet.worksheet(new_worksheet_name)
    else:
        sheet = sheet.add_worksheet(title=new_worksheet_name, rows=10, cols=10)
    set_with_dataframe(sheet, data)


AttributeError: 'Worksheet' object has no attribute 'worksheets'

In [21]:
sheet.title

'vix_data'

In [4]:
fd = pd.read_excel(r"C:\Users\AhmadAizudeen\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Capital Flow Monitor\SEACEN CFM January 2025\edited files\Data\Growth and Inflation 3Q2024.xlsx"
                   , sheet_name=2, nrows=12)


col_name = list(map(convert_date_to_quarter,fd.columns))
fd.rename({fd.columns[0]: 'Region'}, axis=1, inplace=True)
fd.set_index('Region', inplace=True)
fd.loc['Total'] = fd.sum()

def fd_calc(fd: pd.DataFrame, cal = True) -> pd.DataFrame:
    sum_row = fd.sum()
    fd.loc['Total'] = sum_row
    if cal:
        fd = fd.apply(lambda x: x/fd.loc['Total'], axis=1)
    return fd


In [5]:
df2 = df.copy()
df2 = df2.drop([0,1], axis=0)
ch_df = pd.read_excel(r"C:\Users\AhmadAizudeen\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Capital Flow Monitor\SEACEN CFM January 2025\edited files\Data\Growth and Inflation 3Q2024.xlsx"
                   , sheet_name=0, header=1)
ch_df = ch_df.drop([0,1])[['Region', 'China']].set_index('Region').rename(columns={'Region': 'Date'})
df2 = df2[df2.columns[:13]].rename(columns={'Region':'Date'}).set_index('Date')
df2['China'] = ch_df['China']
df3 = df2[['China', 'India', 'Malaysia', 'Mongolia', 'Philippines', 'Taiwan', 'Thailand']].apply(lambda x : ((x/x.shift(4)) - 1)*100)
df3 = pd.concat([df2[['Hong Kong SAR (China)', 'Indonesia', 'South Korea', 'Singapore', 'Vietnam']], df3], axis=1)
df3.reset_index(inplace=True)
df3["Date"] = pd.to_datetime(df3["Date"]) + pd.offsets.MonthEnd(0)
select_row = len(fd.columns)
df4 = df3[-select_row:]


df4['Date'] = list(df4['Date'].apply(lambda x: convert_date_to_quarter(str(x), False)))
df4 = df4.set_index('Date').transpose().reset_index().rename(columns={'index':'Region'}) #this is bullshit
df4.set_index('Region', inplace=True)
df4.rename(index={'Hong Kong SAR (China)': 'Hong Kong', 'South Korea': 'Korea', 'Taiwan': 'Taipei'}, inplace=True)
df4 = df4.reindex(fd.index[:12])


c:\USERS\AHMADAIZUDEEN\ONEDRIVE - THE SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\DESKTOP\NLP PROJECT\VENC\Lib\site-packages\pandas\core\indexes\base.py:7588: FutureWarning: Dtype inference on a pandas object (Series, Index, ExtensionArray) is deprecated. The Index constructor will keep the original dtype in the future. Call `infer_objects` on the result to get the old behavior.
  return Index(sequences[0], name=names)
C:\Users\AhmadAizudeen\AppData\Local\Temp\ipykernel_9924\17278457.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df4['Date'] = list(df4['Date'].apply(lambda x: convert_date_to_quarter(str(x), False)))


In [9]:
asean5 = fd_calc(fd.loc[['Indonesia', 'Malaysia', 'Philippines','Thailand', 'Vietnam']]) * fd_calc(df4.loc[['Indonesia', 'Malaysia', 'Philippines','Thailand', 'Vietnam']], cal=False)
asean5.drop('Total', inplace=True)
asean5_sum = asean5.sum()
asean5.loc['ASEAN-5'] = asean5_sum

adv_asia = fd_calc(fd.loc[['Hong Kong', 'Korea', 'Singapore','Taipei']]) * fd_calc(df4.loc[['Hong Kong', 'Korea', 'Singapore','Taipei']], cal=False)
adv_asia.drop('Total', inplace=True)
adv_asia_sum = adv_asia.sum()
adv_asia.loc['Asia Advanced Economies'] = adv_asia_sum

asia = fd.apply(lambda x: x/fd.loc['Total'], axis=1) * df4
asia_sum = asia.sum()
asia.loc['Asia'] = asia_sum

gdp_growth = pd.concat([asia.loc['Asia'], df4.loc['China'], df4.loc['India'], asean5.loc['ASEAN-5'], adv_asia.loc['Asia Advanced Economies']], axis=1).reset_index().rename(columns={'index':'Quarter'})
gdp_growth = gdp_growth.melt(id_vars="Quarter", var_name="Region", value_name="Value")
gdp_growth

,Quarter,Region,Value
0,1Q2022,Asia,4.552098
1,2Q2022,Asia,4.220563
2,3Q2022,Asia,4.797955
3,4Q2022,Asia,3.267145
4,1Q2023,Asia,4.486541
5,2Q2023,Asia,5.913564
6,3Q2023,Asia,5.195059
7,4Q2023,Asia,5.671215
8,1Q2024,Asia,5.633311
9,2Q2024,Asia,5.063675


In [11]:

gdp_growth['Quarter'].unique()

'1Q2022'